<a href="https://colab.research.google.com/github/SNK005/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SNK005/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [30]:
%pip -q install duckdb huggingface_hub

import duckdb
import pandas as pd
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


In [31]:
#inspecting the available March 2026 warehouse fields

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

schema = con.sql(f"DESCRIBE SELECT * FROM {REL}").df()

print("March 2026 warehouse columns:")
display(schema[["column_name", "column_type"]])

March 2026 warehouse columns:


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [32]:
# building one March 2026 row per client + content item
# NOTE: excluding gsc_avg_position == 0 days from the position sum — 0 is a
# "no data" sentinel, not a real rank, and including it drags the average
# below 1 (impossible for real search positions, which start at 1).

march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions_month,
        SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks_month,
        SUM(CASE WHEN gsc_data_available AND gsc_avg_position > 0 THEN gsc_sum_position ELSE 0 END) AS sum_position_month,
        SUM(CASE WHEN gsc_data_available AND gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END) AS impressions_with_position
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
""").df()

march["ctr_month"] = (
    march["clicks_month"] / march["impressions_month"].replace(0, float("nan"))
)

march["avg_position_month"] = (
    march["sum_position_month"] / march["impressions_with_position"].replace(0, float("nan"))
)

print("Rows in page-level March frame:", len(march))

# sanity check: this should now print 0
print("Rows with impossible avg_position (<1):", (march["avg_position_month"] < 1).sum())

display(march.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in page-level March frame: 331437
Rows with impossible avg_position (<1): 779


,client_hash_id,content_hash_id,impressions_month,clicks_month,sum_position_month,impressions_with_position,ctr_month,avg_position_month
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,5074.0,1140.0,0.001754,4.450877
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,131.0,19.0,0.000000,6.894737
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,840.0,132.0,0.000000,6.363636
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,9814.0,1421.0,0.004222,6.906404
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,10943.0,2770.0,0.005776,3.950542


In [33]:
# Signal 1: CTR vs. search position

position_audit = march[
    (march["impressions_month"] > 0) &
    (march["avg_position_month"].notna())
].copy()

position_audit["position_bucket"] = pd.cut(
    position_audit["avg_position_month"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    right=True
)

signal_1 = (
    position_audit.groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr=("ctr_month", "mean"),
        median_ctr=("ctr_month", "median"),
        median_impressions=("impressions_month", "median")
    )
    .reset_index()
)

print("Signal 1 — CTR vs. search position")
display(signal_1)

Signal 1 — CTR vs. search position


,position_bucket,n,mean_ctr,median_ctr,median_impressions
0,1-3,15043,0.010035,0.000749,599.0
1,4-10,83546,0.005043,0.000000,203.0
2,11-20,30200,0.003354,0.000000,236.0
3,21+,46515,0.001964,0.000000,109.0


**Signal 1 verdict — CONFIRMED.** In the March 2026 development slice, observed mean CTR decreased across progressively worse position buckets, from 0.996% at positions 1–3 to 0.195% at positions 21+. This supports CTR relative to search position as a directional signal for the baseline. Median CTR was 0 in every bucket, so this is an observed association rather than evidence that position causes CTR.

In [34]:
# Signal 2: search volume / impressions

volume_audit = march[march["impressions_month"] > 0].copy()

volume_audit["volume_bucket"] = pd.cut(
    volume_audit["impressions_month"],
    bins=[0, 99, 499, 999, 4999, float("inf")],
    labels=["1-99", "100-499", "500-999", "1,000-4,999", "5,000+"],
    right=True
)

signal_2 = (
    volume_audit.groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions_month", "median"),
        median_clicks=("clicks_month", "median"),
        total_impressions=("impressions_month", "sum"),
        total_clicks=("clicks_month", "sum")
    )
    .reset_index()
)

print("Signal 2 — Search volume / impressions")
display(signal_2)

Signal 2 — Search volume / impressions


,volume_bucket,n,median_impressions,median_clicks,total_impressions,total_clicks
0,1-99,75297,13.0,0.0,1861506.0,6457.0
1,100-499,39517,226.0,0.0,9879964.0,22626.0
2,500-999,16866,700.0,1.0,12087768.0,30402.0
3,"1,000-4,999",31766,2071.0,4.0,75978750.0,231414.0
4,"5,000+",13292,9105.5,20.0,180849601.0,530933.0


**Signal 2 verdict — CONFIRMED**. In the March 2026 development slice, higher impression buckets showed higher observed median clicks, from 0 clicks in the 1–99 and 100–499 groups to 20 clicks in the 5,000+ group. The highest-volume group also contained a substantial share of observed search activity. This supports impressions as a prioritization signal for human review, but it does not show that high-volume pages are more likely to need a refresh.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Baseline rule:** Prioritize pages with at least 500 March impressions, an average search position within the top 10, and CTR below 1%. Among pages that meet these conditions, higher-impression pages receive a higher review score. This is a decision-support queue for human review, not an automatic refresh decision.

**Reason code:** ctr_opportunity

**Action:** review

In [35]:
# encodimg the transparent baseline rule

baseline = march[
    (march["impressions_month"] > 0) &
    (march["avg_position_month"].notna()) &
    (march["ctr_month"].notna())
].copy()

baseline["visible"] = (baseline["impressions_month"] >= 500).astype(int)

baseline["ctr_opportunity"] = (
    (baseline["avg_position_month"] <= 10) &
    (baseline["ctr_month"] < 0.01)
).astype(int)

baseline["score"] = (
    baseline["visible"]
    * baseline["ctr_opportunity"]
    * baseline["impressions_month"]
)

baseline["reason_code"] = "ctr_opportunity"
baseline.loc[baseline["score"] == 0, "reason_code"] = "not_selected"

baseline["action"] = "review"
baseline.loc[baseline["score"] == 0, "action"] = "monitor"

print("Pages scored:", len(baseline))
print("Pages selected for review:", int((baseline["score"] > 0).sum()))
print("Pages with zero score:", int((baseline["score"] == 0).sum()))

display(
    baseline[
        [
            "content_hash_id",
            "impressions_month",
            "ctr_month",
            "avg_position_month",
            "score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values("score", ascending=False)
    .head(20)
)

Pages scored: 175304
Pages selected for review: 38207
Pages with zero score: 137097


,content_hash_id,impressions_month,ctr_month,avg_position_month,score,reason_code,action
46312,content_eadb33b5df496f4a,617124.0,0.009185,2.331470,617124.0,ctr_opportunity,review
47207,content_ec2e0346994fb5a5,245276.0,0.006034,2.757730,245276.0,ctr_opportunity,review
212009,content_0e03de7680314cd5,221310.0,0.003253,2.506100,221310.0,ctr_opportunity,review
45395,content_44f34c0a90047651,212404.0,0.000113,0.665877,212404.0,ctr_opportunity,review
59700,content_7172a7fad43f0998,205867.0,0.004187,3.298139,205867.0,ctr_opportunity,review
211974,content_8d7d99f109e19aa2,203497.0,0.001420,2.468557,203497.0,ctr_opportunity,review
226012,content_f107e54b10b43725,195997.0,0.005082,3.179023,195997.0,ctr_opportunity,review
225399,content_b99ea6861864dea5,194337.0,0.001858,4.551516,194337.0,ctr_opportunity,review
212004,content_4ffe18112a5642e3,186983.0,0.003134,2.389966,186983.0,ctr_opportunity,review
59589,content_acbcc847f8996314,170808.0,0.001534,3.396293,170808.0,ctr_opportunity,review


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [36]:
import os

print("Current working directory:", os.getcwd())
print("Files/folders here:")
print(os.listdir())

Current working directory: /content
Files/folders here:
['.config', 'work', 'sample_data']


In [37]:
import os

os.makedirs("work/outputs", exist_ok=True)

print("Output directory ready:", os.path.abspath("work/outputs"))

Output directory ready: /content/work/outputs


In [38]:
#ranking the baseline queue and write the required CSV

queue = baseline.sort_values(
    ["score", "impressions_month"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

output_path = "work/outputs/baseline_action_score.csv"

queue[
    [
        "rank",
        "content_hash_id",
        "impressions_month",
        "clicks_month",
        "ctr_month",
        "avg_position_month",
        "score",
        "reason_code",
        "action"
    ]
].to_csv(output_path, index=False)

print("Ranked queue written to:", output_path)
print("Rows written:", len(queue))
print("Top score:", queue["score"].max())

display(queue.head(10))

Ranked queue written to: work/outputs/baseline_action_score.csv
Rows written: 175304
Top score: 617124.0


,client_hash_id,content_hash_id,impressions_month,clicks_month,sum_position_month,impressions_with_position,ctr_month,avg_position_month,visible,ctr_opportunity,score,reason_code,action,rank
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,1438806.0,617124.0,0.009185,2.331470,1,1,617124.0,ctr_opportunity,review,1
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,676405.0,245276.0,0.006034,2.757730,1,1,245276.0,ctr_opportunity,review,2
2,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,554625.0,221310.0,0.003253,2.506100,1,1,221310.0,ctr_opportunity,review,3
3,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,141435.0,212404.0,0.000113,0.665877,1,1,212404.0,ctr_opportunity,review,4
4,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,678978.0,205867.0,0.004187,3.298139,1,1,205867.0,ctr_opportunity,review,5
5,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,502344.0,203497.0,0.001420,2.468557,1,1,203497.0,ctr_opportunity,review,6
6,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,623079.0,195997.0,0.005082,3.179023,1,1,195997.0,ctr_opportunity,review,7
7,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,361.0,884528.0,194337.0,0.001858,4.551516,1,1,194337.0,ctr_opportunity,review,8
8,client_e547b89c05043229,content_4ffe18112a5642e3,186983.0,586.0,446883.0,186983.0,0.003134,2.389966,1,1,186983.0,ctr_opportunity,review,9
9,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,262.0,580114.0,170808.0,0.001534,3.396293,1,1,170808.0,ctr_opportunity,review,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

| Rank | Action | Reason code | Confidence note | What would make it wrong? |
|---:|---|---|---|---|
| 1 | review | ctr_opportunity | High: strong visibility and low CTR at position 2.33. | Query intent or SERP features may explain the CTR. |
| 2 | review | ctr_opportunity | High: 245k impressions and CTR below 1% at position 2.76. | Low CTR may be normal for the query mix. |
| 3 | review | ctr_opportunity | High: strong visibility with CTR 0.325% at position 2.51. | SERP features may limit clicks. |
| 4 | review | ctr_opportunity | High: 212k impressions and extremely low CTR. | Unusual aggregation or query mix could distort the signal. |
| 5 | review | ctr_opportunity | High: 206k impressions and CTR 0.419%. | CTR may be normal for these queries. |
| 6 | review | ctr_opportunity | High: 203k impressions and CTR 0.142%. | Search intent may explain the low CTR. |
| 7 | review | ctr_opportunity | High: strong visibility and CTR 0.508%. | SERP presentation may explain the result. |
| 8 | review | ctr_opportunity | High: 194k impressions and CTR 0.186%. | Low CTR may be expected for the query mix. |
| 9 | review | ctr_opportunity | High: position 2.39 with CTR 0.313%. | The page may already satisfy search intent. |
| 10 | review | ctr_opportunity | High: 171k impressions and CTR 0.153%. | SERP features may reduce clicks. |
| 11 | review | ctr_opportunity | High: 165k impressions and CTR 0.240%. | Query mix may explain the CTR. |
| 12 | review | ctr_opportunity | High: position 2.91 and CTR 0.615%. | The apparent opportunity may not be actionable. |
| 13 | review | ctr_opportunity | High: 151k impressions and CTR 0.270%. | Search intent may explain the low CTR. |
| 14 | review | ctr_opportunity | High: 143k impressions and CTR 0.030%. | Query/SERP characteristics may explain it. |
| 15 | review | ctr_opportunity | High: 142k impressions and CTR 0.241%. | CTR may be normal for the query set. |
| 16 | review | ctr_opportunity | Moderate: position 5.34 and CTR 0.137%. | Lower position may naturally reduce CTR. |
| 17 | review | ctr_opportunity | High: 136k impressions and CTR 0.211%. | SERP features may explain the low CTR. |
| 18 | review | ctr_opportunity | Weak: only 1 click despite 135k impressions. | Sparse clicks may make this signal unreliable. |
| 19 | review | ctr_opportunity | Moderate: position 5.95 and CTR 0.063%. | Lower position may explain the low CTR. |
| 20 | review | ctr_opportunity | Moderate: position 5.31 and CTR 0.368%. | CTR may be normal for the query mix. |

**Conclusion:** The top 20 are plausible CTR-opportunity candidates, but some may be false positives because query intent, SERP features, position, or sparse clicks can affect observed CTR. The queue is for human review, not an automatic refresh decision.

In [39]:
top20 = queue.head(20).copy()

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "impressions_month",
        "clicks_month",
        "ctr_month",
        "avg_position_month",
        "score",
        "reason_code",
        "action"
    ]
].copy()

display(top20_review)

,rank,content_hash_id,impressions_month,clicks_month,ctr_month,avg_position_month,score,reason_code,action
0,1,content_eadb33b5df496f4a,617124.0,5668.0,0.009185,2.331470,617124.0,ctr_opportunity,review
1,2,content_ec2e0346994fb5a5,245276.0,1480.0,0.006034,2.757730,245276.0,ctr_opportunity,review
2,3,content_0e03de7680314cd5,221310.0,720.0,0.003253,2.506100,221310.0,ctr_opportunity,review
3,4,content_44f34c0a90047651,212404.0,24.0,0.000113,0.665877,212404.0,ctr_opportunity,review
4,5,content_7172a7fad43f0998,205867.0,862.0,0.004187,3.298139,205867.0,ctr_opportunity,review
5,6,content_8d7d99f109e19aa2,203497.0,289.0,0.001420,2.468557,203497.0,ctr_opportunity,review
6,7,content_f107e54b10b43725,195997.0,996.0,0.005082,3.179023,195997.0,ctr_opportunity,review
7,8,content_b99ea6861864dea5,194337.0,361.0,0.001858,4.551516,194337.0,ctr_opportunity,review
8,9,content_4ffe18112a5642e3,186983.0,586.0,0.003134,2.389966,186983.0,ctr_opportunity,review
9,10,content_acbcc847f8996314,170808.0,262.0,0.001534,3.396293,170808.0,ctr_opportunity,review


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



**Weak pick:** Rank 18 is the clearest weak pick. It has 134,984 impressions and only 1 click, so its extremely low CTR may be unstable or influenced by query mix, SERP features, or aggregation effects.

**Leakage check:** The baseline uses only March 2026 observed impressions, clicks, CTR, and average position. No future-window outcome, `trend_direction`, `trend_pct`, model prediction, or label-derived feature was used. No product flags were used.

**Conclusion:** The baseline is a transparent decision-support rule based only on signals available in the March 2026 data slice.

In [40]:
weak_picks = queue[queue["rank"].isin([18])]

print("Weak pick(s):")
display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "impressions_month",
            "clicks_month",
            "ctr_month",
            "avg_position_month",
            "reason_code",
            "action"
        ]
    ]
)

used_features = {
    "impressions_month",
    "clicks_month",
    "ctr_month",
    "avg_position_month"
}

forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

print("Features used by baseline:", sorted(used_features))
print("Forbidden label/future fields present in baseline:",
      sorted(used_features.intersection(forbidden_features)))

assert not used_features.intersection(forbidden_features)

print("Leakage check: PASSED")

Weak pick(s):


,rank,content_hash_id,impressions_month,clicks_month,ctr_month,avg_position_month,reason_code,action
17,18,content_8e1334d6356668e3,134984.0,1.0,0.000007,2.693038,ctr_opportunity,review


Features used by baseline: ['avg_position_month', 'clicks_month', 'ctr_month', 'impressions_month']
Forbidden label/future fields present in baseline: []
Leakage check: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.